# Imports 

In [30]:
import pandas as pd
import numpy as np
import joblib
import os
import seaborn as sns
import matplotlib.pyplot as plt
import itertools
from scipy.stats import t
import copy 
import pprint
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_selection import r_regression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet, BayesianRidge
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score



In [31]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.codebase import Regressor

In [32]:
baseline=Regressor()

path_to_dev = "../data/assignment1_dev_set.csv"
path_to_val = "../data/assignment1_val_set.csv"

dev_set_df=baseline.load_data(path_to_dev)
print(dev_set_df)
val_set_df=baseline.load_data(path_to_val)
print(val_set_df)

Data for development:
     Unnamed: 0   Project ID Experiment type     Sex  Host age    BMI  \
0             0   PRJEB11419    Metagenomics    Male      53.0  19.01   
1             1  PRJNA388263    Metagenomics  Female      21.0  23.50   
2             2  PRJNA388263    Metagenomics    Male      52.0  25.80   
3             3   PRJEB11419    Metagenomics  Female      40.0  23.49   
4             4   PRJEB11419    Metagenomics  Female      30.0  22.60   
..          ...          ...             ...     ...       ...    ...   
484         484  PRJNA397219    Metagenomics    Male      60.0  24.97   
485         485  PRJNA388263    Metagenomics    Male      52.0  25.80   
486         486  PRJNA397219    Metagenomics    Male      78.0  29.53   
487         487   PRJEB11419    Metagenomics    Male      43.0  25.55   
488         488  PRJNA485797    Metagenomics  Female      51.0  22.59   

    Disease MESH ID  Acholeplasma axanthum  Acidaminococcus fermentans  \
0           D006262        

In [33]:
path_to_final_dev = "../data/development_final_data.csv"
path_to_final_val = "../data/evaluation_final_data.csv"

dev_reduced_df=baseline.preprocess_data(dev_set_df, path_to_final_dev, columns_to_drop=['Unnamed: 0', 'Project ID', 'Experiment type', 'Disease MESH ID'], scale=False)
print(dev_reduced_df)
val_reduced_df=baseline.preprocess_data(val_set_df, path_to_final_val, columns_to_drop=['Unnamed: 0', 'Project ID', 'Experiment type', 'Disease MESH ID'], scale=False)
print(val_reduced_df)

The preprocessed data was saved to ../data/development_final_data.csv.
     Sex  Host age    BMI  Acholeplasma axanthum  Acidaminococcus fermentans  \
0      1      53.0  19.01               0.000000                    0.000000   
1      0      21.0  23.50               0.001028                    0.000000   
2      1      52.0  25.80               0.001406                    0.000000   
3      0      40.0  23.49               0.000000                    0.008825   
4      0      30.0  22.60               0.002878                    0.037419   
..   ...       ...    ...                    ...                         ...   
484    1      60.0  24.97               0.000000                    0.000000   
485    1      52.0  25.80               0.000000                    0.000000   
486    1      78.0  29.53               0.000000                    0.000000   
487    1      43.0  25.55               0.001556                    0.001556   
488    0      51.0  22.59               0.000000 

## Baseline: No feature selection, no model-tuning

In [34]:
baseline=Regressor()
X, y = baseline.separate_features_target(dev_reduced_df, target='BMI', columns_to_remove=['Sex', 'Host age'])
print(X)
print(y)

     Acholeplasma axanthum  Acidaminococcus fermentans  \
0                 0.000000                    0.000000   
1                 0.001028                    0.000000   
2                 0.001406                    0.000000   
3                 0.000000                    0.008825   
4                 0.002878                    0.037419   
..                     ...                         ...   
484               0.000000                    0.000000   
485               0.000000                    0.000000   
486               0.000000                    0.000000   
487               0.001556                    0.001556   
488               0.000000                    0.011382   

     Acidaminococcus intestini  Actinomyces lingnae  Akkermansia muciniphila  \
0                     0.000000             0.000000                 0.017674   
1                     0.000000             0.000000                13.015800   
2                     0.000000             0.001406            

In [35]:
print('Results from baseline model, where no feature selection nor model-tuning took place:')
baseline_results=baseline.train_models(X, y)

Results from baseline model, where no feature selection nor model-tuning took place:
enet RMSE: 3.8141
svr RMSE: 3.7522
breg RMSE: 3.8384


In [36]:
# Since no feature selection took place for this part of our model analysis, therewas no need for evaluation set alignment to the developmet set
# Features and target have already been separated 
baseline = Regressor()


# model_instance-> ElasticNet with default parameters and no feauture selection
enet_results, enet_metrics = baseline.evaluate_model(ElasticNet(), X, y, runs=30, test_size=0.2, save_path=None)
nofs_nomt_enet_df_summary = pd.DataFrame.from_dict(enet_results, orient='index')
print('Evaluation Metric results from baseline ElasticNet model:')
print(nofs_nomt_enet_df_summary)

# model_instance-> SVR with default parameters and no feauture selection
svr_results, svr_metrics = baseline.evaluate_model(SVR(), X, y, runs=30, test_size=0.2, save_path=None)
nofs_nomt_svr_df_summary = pd.DataFrame.from_dict(svr_results, orient='index')
print('Evaluation Metric results from baseline SVR model:')
print(nofs_nomt_svr_df_summary)

# model_instance-> SVR with default parameters and no feauture selection
breg_results, breg_metrics = baseline.evaluate_model(BayesianRidge(), X, y, runs=30, test_size=0.2, save_path=None)
nofs_nomt_breg_df_summary = pd.DataFrame.from_dict(breg_results, orient='index')
print('Evaluation Metric results from baseline Bayesianregression model:')
print(nofs_nomt_breg_df_summary)


AttributeError: module 'joblib' has no attribute 'loads'

## Feature Selection, No Model-Tuning

In [ ]:
selected_features, correlations=baseline.select_features(X, y, threshold=0.1)
print(selected_features)

X_selected=X[selected_features]
print(X_selected)

selected_feature_names = X_selected.columns.tolist()
target = 'BMI'
dev_selected_df = dev_reduced_df[selected_feature_names + [target]]
print(dev_selected_df)

The selected features of 134 were: 11
Index(['Alistipes putredinis', 'Christensenella minuta',
       'Desulfonispora thiosulfatigenes', 'Ruminiclostridium thermocellum',
       'Ruminococcus champanellensis', 'Ruminococcus flavefaciens',
       'Sporobacter termitidis', 'Clostridium clariflavum',
       'Clostridium stercorarium', 'Clostridium symbiosum',
       'Eubacterium sulci'],
      dtype='object')
     Alistipes putredinis  Christensenella minuta  \
0                0.005891                0.000000   
1                0.403916                3.756010   
2                0.105459                0.154673   
3                0.586834                0.017649   
4                1.004550                0.031662   
..                    ...                     ...   
484              2.620060                0.000000   
485              0.226497                0.013959   
486              1.722580                0.142391   
487              0.455776                0.015555   
488    